# 03 — Train, Validation & Test

**Sprint 7 — Machine Learning Fundamentals for AI/ML Engineers**
**Notebook 3 of 20**

This notebook covers the single most important discipline in applied ML: **how you split your data
decides whether your evaluation numbers can be trusted at all.** Get this wrong and every metric you
report later — accuracy, RMSE, F1 — is quietly lying to you.

**Datasets used for the code demos:**
- **`diabetes`** (`sklearn.datasets.load_diabetes`, regression, fully bundled with scikit-learn — no
  network download required, unlike `fetch_california_housing`) for the train/test-split and
  cross-validation demos. Everything below is written so you can swap in
  `fetch_california_housing(as_frame=True)` in place of `load_diabetes(as_frame=True)` on your own
  machine (where the download works fine) and re-run this notebook unchanged against your housing
  project's own dataset — the mechanics (splitting, CV, leakage, stratification) don't depend on which
  dataset you use.
- **`digits`** (`sklearn.datasets.load_digits`, classification, also bundled — an 8x8-pixel little
  sibling of the full MNIST set you used in Task 2) for the Stratified K-Fold demo, since it needs no
  network download and keeps this notebook fast to run. Everything shown transfers directly to the full
  `mnist_784` dataset if you want to re-run it there.


## 1. Why can't we just train and evaluate on the same data?

If you fit a model on the data and then also *score* it on that exact same data, a model that has
simply **memorized** every training example (rather than learned a generalizable pattern) will score
perfectly — and tell you nothing about how it will behave on a new patient's data or a new handwritten
digit it hasn't seen. The entire point of evaluation is to estimate performance on data the
model has *never seen*, because that's the only situation that resembles production use.

This is why we split the data into separate, non-overlapping subsets **before** training even starts.


In [1]:
from sklearn.datasets import load_diabetes
import pandas as pd

diabetes = load_diabetes(as_frame=True)
X, y = diabetes.data, diabetes.target
print("Features (X):", list(X.columns))
print("Target (y):   a quantitative measure of disease progression one year after baseline")
print("Shape:", X.shape, y.shape)
X.head()


Features (X): ['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']
Target (y):   a quantitative measure of disease progression one year after baseline
Shape: (442, 10) (442,)


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641


## 2. Training, Validation, and Test datasets

| Set | Used for | Touched how often |
|---|---|---|
| **Training set** | Fitting the model's parameters (what `fit()` actually learns from) | Every training run |
| **Validation set** | Comparing candidate models / tuning hyperparameters, deciding *which* configuration to keep | Repeatedly, during development |
| **Test set** | The final, honest estimate of real-world performance | **Exactly once**, at the very end |

The reason validation and test are *separate* sets (not just one held-out set): if you tune
hyperparameters by repeatedly checking performance on the same held-out data, you are — indirectly —
fitting to that data too (you keep the settings that happen to score well on it). The validation set
absorbs that repeated peeking; the test set stays untouched so it still gives an unbiased final answer.

**Typical splits:**
- Simple: 80% train / 20% test (no separate validation set — used when you'll use cross-validation
  instead of a fixed validation split, see Section 4).
- Three-way: 60% train / 20% validation / 20% test, or similar, when you want a fixed validation set
  rather than cross-validation.


In [2]:
from sklearn.model_selection import train_test_split

# --- Simple two-way split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Two-way split:")
print(f"  train: {X_train.shape[0]} rows ({X_train.shape[0]/len(X):.0%})")
print(f"  test:  {X_test.shape[0]} rows ({X_test.shape[0]/len(X):.0%})")

# --- Three-way split: do it in two steps -- split off test first, then split train into train/val
X_train_full, X_test2, y_train_full, y_test2 = train_test_split(X, y, test_size=0.2, random_state=42)
X_train2, X_val, y_train2, y_val = train_test_split(X_train_full, y_train_full, test_size=0.25, random_state=42)
# 0.25 of the remaining 80% = 20% of the original data -> final split is 60/20/20
print("\nThree-way split:")
print(f"  train: {X_train2.shape[0]} rows ({X_train2.shape[0]/len(X):.0%})")
print(f"  val:   {X_val.shape[0]} rows ({X_val.shape[0]/len(X):.0%})")
print(f"  test:  {X_test2.shape[0]} rows ({X_test2.shape[0]/len(X):.0%})")


Two-way split:
  train: 353 rows (80%)
  test:  89 rows (20%)

Three-way split:
  train: 264 rows (60%)
  val:   89 rows (20%)
  test:  89 rows (20%)


## 3. `random_state` — why we pin it

`train_test_split` shuffles before splitting. Without fixing the randomness, you'd get a **different**
split every time you re-run the notebook — meaning your reported metric would wobble run to run for
reasons that have nothing to do with your model, only to do with which rows happened to land in the
test set. Setting `random_state=<any fixed integer>` makes the split **reproducible**: same input data
+ same `random_state` → identical split, every time, on any machine. This matters for debugging,
comparing models fairly (they must see the *same* test set), and for other people being able to
reproduce your results.


In [3]:
# Same random_state -> identical split, every time
split_a = train_test_split(X, y, test_size=0.2, random_state=42)
split_b = train_test_split(X, y, test_size=0.2, random_state=42)
print("Identical split with the same random_state?", (split_a[0].index == split_b[0].index).all())

# Different random_state -> different split
split_c = train_test_split(X, y, test_size=0.2, random_state=7)
print("Identical split with a different random_state?", (split_a[0].index == split_c[0].index).all())


Identical split with the same random_state? True
Identical split with a different random_state? False


## 4. Cross Validation

A single train/test (or train/val) split has a downside: your performance estimate depends on exactly
*which* rows happened to land in the held-out set — an unlucky split can make a good model look bad, or
vice versa, purely by chance (this is worse the smaller your dataset is).

**K-Fold Cross Validation** fixes this by splitting the training data into **K equal folds**. The model
is trained K times: each time, one fold is held out for validation and the other K-1 folds are used for
training. You end up with K performance scores, and you report their **mean and standard deviation** —
the mean gives a more stable estimate than any single split, and the standard deviation tells you how
*sensitive* the model's performance is to which data it saw (a large std = an unstable model or too
little data).

Common choice: **K = 5** or **K = 10** — high enough to average out split-luck, low enough to stay
computationally affordable (K folds = K full training runs).


In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold
import numpy as np

model = LinearRegression()

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
# scoring='neg_root_mean_squared_error' because cross_val_score maximizes by convention,
# so error metrics are negated -- we flip the sign back for a readable RMSE.
scores = cross_val_score(model, X_train_full, y_train_full, cv=kfold, scoring="neg_root_mean_squared_error")
rmse_scores = -scores

print("RMSE per fold:", np.round(rmse_scores, 4))
print(f"Mean RMSE:  {rmse_scores.mean():.4f}")
print(f"Std  RMSE:  {rmse_scores.std():.4f}   <- how much the estimate wobbles across folds")


RMSE per fold: [53.3737 56.761  53.0431 54.46   59.3353]
Mean RMSE:  55.3946
Std  RMSE:  2.3615   <- how much the estimate wobbles across folds


## 5. Stratified K-Fold

Plain K-Fold shuffles rows randomly into folds. For **classification**, if one class is rarer than
others, a random fold can end up with very few (or zero) examples of that class purely by chance —
distorting both training and evaluation for that fold. **Stratified K-Fold** fixes this by preserving
the **same class proportions in every fold** as in the full dataset — exactly the reason we used a
stratified split (on an income-category bin) for the California Housing test set in your earlier
regression project, and why it matters even more here with a true multi-class target.


In [5]:
from sklearn.datasets import load_digits
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd

digits = load_digits()
Xd, yd = digits.data, digits.target
print("digits dataset:", Xd.shape, "-- classes:", np.unique(yd))
print("Class counts:", pd.Series(yd).value_counts().sort_index().to_dict())

clf = LogisticRegression(max_iter=2000)

plain_kfold = KFold(n_splits=5, shuffle=True, random_state=42)
strat_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

plain_scores = cross_val_score(clf, Xd, yd, cv=plain_kfold, scoring="accuracy")
strat_scores = cross_val_score(clf, Xd, yd, cv=strat_kfold, scoring="accuracy")

print(f"\nPlain KFold      accuracy: mean={plain_scores.mean():.4f}  std={plain_scores.std():.4f}")
print(f"Stratified KFold accuracy: mean={strat_scores.mean():.4f}  std={strat_scores.std():.4f}")

# Show WHY it matters: check class balance actually achieved within one fold of each strategy
fold_plain = next(plain_kfold.split(Xd, yd))
fold_strat = next(strat_kfold.split(Xd, yd))
print("\nClass proportions in the held-out fold, plain KFold:     ",
      np.round(pd.Series(yd[fold_plain[1]]).value_counts(normalize=True).sort_index().values, 3))
print("Class proportions in the held-out fold, Stratified KFold:",
      np.round(pd.Series(yd[fold_strat[1]]).value_counts(normalize=True).sort_index().values, 3))
print("Class proportions in the FULL dataset:                   ",
      np.round(pd.Series(yd).value_counts(normalize=True).sort_index().values, 3))


digits dataset: (1797, 64) -- classes: [0 1 2 3 4 5 6 7 8 9]
Class counts: {0: 178, 1: 182, 2: 177, 3: 183, 4: 181, 5: 182, 6: 181, 7: 179, 8: 174, 9: 180}



Plain KFold      accuracy: mean=0.9633  std=0.0060
Stratified KFold accuracy: mean=0.9666  std=0.0046

Class proportions in the held-out fold, plain KFold:      [0.092 0.078 0.092 0.094 0.128 0.131 0.097 0.094 0.083 0.111]
Class proportions in the held-out fold, Stratified KFold: [0.1   0.1   0.1   0.1   0.103 0.1   0.1   0.1   0.097 0.1  ]
Class proportions in the FULL dataset:                    [0.099 0.101 0.098 0.102 0.101 0.101 0.101 0.1   0.097 0.1  ]


**Reading the output:** the digits dataset happens to be fairly balanced already, so plain
K-Fold isn't dramatically wrong here — but notice how much closer the Stratified fold's class
proportions track the full dataset's proportions. On an imbalanced dataset (e.g., 95% of one class),
plain K-Fold could easily produce a fold with almost no minority-class examples, while Stratified K-Fold
never lets that happen by construction. **Default to Stratified K-Fold for classification** unless you
have a specific reason not to.


## 6. Data Leakage

**Data leakage** happens when information from outside the training set — often, information that
implicitly comes from the validation or test set — influences model training, making performance look
better than it will actually be in production.

Common ways leakage sneaks in:
- **Preprocessing before splitting**: fitting a `StandardScaler` (or an imputer, or a feature selector)
  on the *entire* dataset before doing `train_test_split`. The scaler's mean/std then "saw" the test
  data, so test-set information has leaked into how the training data is represented.
- **Target leakage**: including a feature that is only available *because* the outcome already
  happened (e.g., using "was a refund issued" as a feature to predict "is this transaction fraudulent" —
  refunds happen *after* fraud is confirmed).
- **Group leakage**: rows from the same real-world entity (the same patient, the same customer) ending
  up in both train and test, letting the model partly "recognize" a test row instead of generalizing.
- **Temporal leakage**: for time-ordered data, randomly shuffling into train/test lets the model train
  on future data to predict the past — production would never have that future data available.

The fix for the most common case (preprocessing) is always the same rule: **fit every preprocessing
step ONLY on the training data, then apply (`transform`, never re-fit) that same fitted transformer to
the validation/test data.** A scikit-learn `Pipeline` enforces this automatically when used correctly
with cross-validation.


In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import root_mean_squared_error
import numpy as np

# ---------- WRONG: scaler fitted on the full dataset BEFORE splitting ----------
scaler_leaky = StandardScaler().fit(X)          # <- sees ALL rows, including future test rows
X_scaled_leaky = scaler_leaky.transform(X)
Xtr_l, Xte_l, ytr_l, yte_l = train_test_split(X_scaled_leaky, y, test_size=0.2, random_state=42)

leaky_model = LinearRegression().fit(Xtr_l, ytr_l)
leaky_rmse = root_mean_squared_error(yte_l, leaky_model.predict(Xte_l))

# ---------- CORRECT: split first, fit the scaler on train only, transform test with it ----------
Xtr_c, Xte_c, ytr_c, yte_c = train_test_split(X, y, test_size=0.2, random_state=42)
pipeline = Pipeline([
    ("scaler", StandardScaler()),     # fit_transform() is called ONLY on X_train inside .fit()
    ("model", LinearRegression()),
])
pipeline.fit(Xtr_c, ytr_c)             # scaler is fitted on Xtr_c only
correct_rmse = root_mean_squared_error(yte_c, pipeline.predict(Xte_c))

print(f"Leaky-preprocessing RMSE:   {leaky_rmse:.4f}")
print(f"Correct pipeline RMSE:      {correct_rmse:.4f}")
print(f"Difference:                 {leaky_rmse - correct_rmse:.6f}")


Leaky-preprocessing RMSE:   53.8534
Correct pipeline RMSE:      53.8534
Difference:                 0.000000


**Reading this output is itself an important lesson.** The two RMSE values come out
*identical*. That is not a mistake in the demo — it's a mathematical fact: **Ordinary Least Squares
Linear Regression's predictions are invariant to per-feature affine scaling.** Rescaling every feature
by a constant factor and shift just rescales the corresponding coefficients to compensate; the actual
*predictions* the line makes don't change. So for this specific model, this specific leak happens to be
numerically harmless.

**That does not make the leak safe** — it means this particular combination (linear regression + a pure
scaling transform) is one of the rare cases where leakage doesn't show up numerically. Swap in a model
whose output genuinely depends on the exact scaled values — a distance-based algorithm like KNN, or an
SVM, or a *regularized* linear model (Ridge/Lasso, where the penalty term is scale-dependent) — and the
leaked statistics change the result:


In [7]:
from sklearn.neighbors import KNeighborsRegressor

# Same leaky-vs-correct comparison, but with a model whose predictions DO depend on the exact scaling
knn_leaky = KNeighborsRegressor(n_neighbors=5).fit(Xtr_l, ytr_l)
knn_leaky_rmse = root_mean_squared_error(yte_l, knn_leaky.predict(Xte_l))

knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsRegressor(n_neighbors=5)),
])
knn_pipeline.fit(Xtr_c, ytr_c)
knn_correct_rmse = root_mean_squared_error(yte_c, knn_pipeline.predict(Xte_c))

print(f"KNN, leaky-preprocessing RMSE:  {knn_leaky_rmse:.4f}")
print(f"KNN, correct pipeline RMSE:     {knn_correct_rmse:.4f}")
print(f"Difference:                     {knn_leaky_rmse - knn_correct_rmse:.4f}")
print("\nNow there IS a gap -- the fitted scaler saw 442 rows (including the 89 test rows) in the leaky")
print("version, vs. only the 353 training rows in the correct version, so every training point ends up")
print("scaled slightly differently, which shifts which neighbors are 'closest' to each query point.")
print("On messier real-world data, or with imputation/feature-selection leaks instead of just scaling,")
print("this gap grows -- and it ALWAYS biases you toward an overly optimistic number, never a pessimistic")
print("one, because the leaked information only ever helps the model, it can't hurt it.")


KNN, leaky-preprocessing RMSE:  54.9461
KNN, correct pipeline RMSE:     55.2037
Difference:                     -0.2576

Now there IS a gap -- the fitted scaler saw 442 rows (including the 89 test rows) in the leaky
version, vs. only the 353 training rows in the correct version, so every training point ends up
scaled slightly differently, which shifts which neighbors are 'closest' to each query point.
On messier real-world data, or with imputation/feature-selection leaks instead of just scaling,
this gap grows -- and it ALWAYS biases you toward an overly optimistic number, never a pessimistic
one, because the leaked information only ever helps the model, it can't hurt it.


## 7. Overfitting & Underfitting (introduction)

Two failure modes that the train/validation/test discipline exists to catch — the full treatment,
including the bias-variance tradeoff, is **Notebook 15**, but you need the vocabulary now:

- **Underfitting**: the model is too simple to capture the real pattern in the data. It performs
  poorly on *both* the training set and the validation/test set.
- **Overfitting**: the model has essentially memorized the training set, including its noise, rather
  than learning the general pattern. It performs very well on the training set but noticeably worse on
  validation/test data — this gap is the telltale sign.
- **Good fit**: training and validation performance are both reasonably strong and reasonably close to
  each other.

You *detect* overfitting/underfitting precisely by comparing training-set performance to
validation-set performance — which is exactly why the split disciplines in this notebook exist.


In [8]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_error

for max_depth, label in [(2, "very shallow (likely UNDERFIT)"),
                          (None, "unlimited depth (likely OVERFIT)")]:
    tree = DecisionTreeRegressor(max_depth=max_depth, random_state=42)
    tree.fit(X_train2, y_train2)
    train_rmse = root_mean_squared_error(y_train2, tree.predict(X_train2))
    val_rmse = root_mean_squared_error(y_val, tree.predict(X_val))
    gap = val_rmse - train_rmse
    print(f"max_depth={str(max_depth):>5}  ({label})")
    print(f"    train RMSE = {train_rmse:.4f}   val RMSE = {val_rmse:.4f}   gap = {gap:.4f}\n")


max_depth=    2  (very shallow (likely UNDERFIT))
    train RMSE = 57.6273   val RMSE = 58.7814   gap = 1.1541

max_depth= None  (unlimited depth (likely OVERFIT))
    train RMSE = 0.0000   val RMSE = 65.8088   gap = 65.8088



**Reading the output:** the shallow tree (`max_depth=2`) has train and validation RMSE close
together but both relatively high — it hasn't learned enough structure (underfit). The unconstrained
tree (`max_depth=None`) drives training RMSE very low (near-perfect memorization of the training rows)
while validation RMSE is much higher — a large train/val gap is the signature of overfitting. Notebook
15 covers how to fix both (regularization, pruning, more data, simpler/more complex models).


## 8. Why the test set must never be used for tuning

Putting it all together — here is the discipline this whole notebook has been building toward:

1. Split off the **test set** first, before looking at anything else. Set it aside.
2. Use **only** the training data (directly, with a validation split, or with cross-validation) to
   explore features, compare models, and tune hyperparameters.
3. Once you've locked in a final model and its hyperparameters — no more changes — evaluate it on the
   test set **exactly once**.
4. That number is your honest, reportable estimate of real-world performance.

If you instead use the test set to decide *anything* (which model to pick, which hyperparameters to
use, whether to try a different feature), you are — even unintentionally — tuning to the test set. Each
peek makes your final "unbiased" number a little more biased, because you're now selecting for
whatever happens to score well on that specific held-out data, which is precisely what the test set was
supposed to help you avoid. This is the ML equivalent of grading your own exam using the answer key
*while* you're still allowed to change your answers.


## 9. Summary — Key Takeaways

- Train on the training set, tune using validation (a fixed split or, better with limited data,
  cross-validation), and touch the test set exactly once, at the very end.
- `random_state` makes splits reproducible — always set it explicitly.
- K-Fold Cross Validation trades more compute for a more stable, less split-luck-dependent performance
  estimate; report mean **and** standard deviation across folds.
- Use **Stratified** K-Fold by default for classification so every fold reflects the true class balance.
- Data leakage (fitting preprocessing before splitting, target leakage, group leakage, temporal
  leakage) makes your evaluation numbers optimistic and untrustworthy — always fit transformers on
  training data only, ideally inside a `Pipeline`.
- A large gap between training and validation performance signals overfitting; poor performance on
  both signals underfitting.


## 10. Try It Yourself (no solutions given)

1. Re-run Section 4's cross-validation with `n_splits=3` and `n_splits=10` instead of 5. How does the
   mean RMSE change? How does the standard deviation change? Can you explain why, in terms of how much
   data each fold's training set now has?
2. In Section 6, deliberately introduce a *worse* leakage bug: fit a `SimpleImputer` (mean strategy) on
   the whole dataset before splitting, on a copy of `X` where you've manually set 5% of the `bmi` column
   values to `NaN`. Compare the leaky vs. correct RMSE — is the gap bigger or smaller than the scaling
   example above, and why might that be?
3. Using `load_digits()`, deliberately create an **imbalanced** version of the dataset (e.g., keep only
   10% of the examples for digit `5`, but all examples for every other digit). Re-run the plain-KFold
   vs. Stratified-KFold comparison from Section 5 on this imbalanced version — does the gap between the
   two strategies become more visible now?
4. In your own words: why is it wrong to pick your final model based on which one scores best on the
   test set, even if you only check the test set once *per candidate model* (not per hyperparameter)?
